[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-10-testing-testclient.ipynb#scrollTo=a1b2c3d4)

---
# Day 10 · Testing with TestClient and pytest
**certified-journeys / fastapi-certified** · Review Day

> **Goal for today:** Write a complete test suite for a FastAPI app — synchronous and async — using TestClient, dependency overrides, and pytest fixtures so every route is verified in isolation.


In [ ]:
%pip install -q fastapi httpx pytest pytest-asyncio anyio


## Step 1 · Why TestClient, not requests?

FastAPI ships with `TestClient` (re-exported from Starlette), which wraps `httpx` under the hood.
It starts the app **in-process** — no network socket, no server process — so tests run instantly.

```
┌──────────────────────────────────────────┐
│  pytest test_main.py                     │
│  ├─ TestClient(app)                      │
│  │   └─ httpx.Client (ASGI transport)   │
│  │       └─ your FastAPI app (in-proc)  │
└──────────────────────────────────────────┘
```

Key facts:
- **No real HTTP** — requests go through ASGI transport, not TCP.
- **Lifespan events fire** — `startup` / `shutdown` handlers run.
- **Session-scoped** — wrap in `with TestClient(app) as client:` to trigger lifespan.


In [ ]:
# ── Build a minimal FastAPI app we'll test throughout this notebook ──────────
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Dict

app = FastAPI()

# In-memory store (acts like a tiny database)
fake_db: Dict[int, dict] = {}
next_id = 1

class ItemIn(BaseModel):
    name: str
    price: float

class ItemOut(ItemIn):
    id: int

@app.get("/")
def read_root():
    return {"message": "FastAPI is running"}

@app.get("/items/{item_id}", response_model=ItemOut)
def get_item(item_id: int):
    if item_id not in fake_db:
        raise HTTPException(status_code=404, detail="Item not found")
    return fake_db[item_id]

@app.post("/items", response_model=ItemOut, status_code=201)
def create_item(item: ItemIn):
    global next_id
    # Reject duplicate names
    for existing in fake_db.values():
        if existing["name"] == item.name:
            raise HTTPException(status_code=409, detail="Item already exists")
    record = {"id": next_id, "name": item.name, "price": item.price}
    fake_db[next_id] = record
    next_id += 1
    return record

print("App defined with routes: GET /, GET /items/{id}, POST /items")


### What just happened?
- Defined a FastAPI app with **three routes** and an in-memory dict as the store.
- **`ItemIn` / `ItemOut`** are Pydantic models — FastAPI validates request bodies and serialises responses automatically.
- The `409 Conflict` on duplicate names gives us an interesting test case beyond the happy path.
- `fake_db` and `next_id` are module-level globals; we'll reset them in pytest fixtures.


## Step 2 · First test — GET /

The simplest test verifies the root endpoint returns the expected JSON and status code.
We instantiate `TestClient` with our app, call `.get()`, and assert on the response.

Key `TestClient` methods mirror `requests`:

| Method | Use |
|--------|-----|
| `.get(url)` | HTTP GET |
| `.post(url, json=...)` | HTTP POST with JSON body |
| `.put / .patch / .delete` | same pattern |
| `.response.status_code` | integer status |
| `.response.json()` | parsed body |


In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# ── Test GET / ───────────────────────────────────────────────────────────────
def test_read_root():
    response = client.get("/")
    assert response.status_code == 200
    assert response.json() == {"message": "FastAPI is running"}
    print("PASSED: GET / returns 200 with correct message")

test_read_root()


### What just happened?
- `TestClient(app)` creates an httpx client pointed at the ASGI app.
- `.get("/")` dispatches directly into FastAPI — **no socket needed**.
- We called the test function inline to see output in Colab; in a real project these would be collected by `pytest`.
- **`response.json()`** parses the JSON body — always prefer this over `response.text` in assertions.


## Step 3 · POST /items — happy path, validation error, duplicate

Good test suites cover **three cases** for write endpoints:

| Case | Trigger | Expected status |
|------|---------|----------------|
| Happy path | valid body | 201 Created |
| Validation error | missing/wrong-type field | 422 Unprocessable Entity |
| Business rule | duplicate name | 409 Conflict |

FastAPI's Pydantic integration automatically returns 422 for schema violations — you never have to write that handler.


In [ ]:
import importlib, sys

# Reset in-memory store between tests
def reset_db():
    global fake_db, next_id
    fake_db.clear()
    next_id = 1

# ── Happy path ───────────────────────────────────────────────────────────────
def test_create_item_happy():
    reset_db()
    response = client.post("/items", json={"name": "Widget", "price": 9.99})
    assert response.status_code == 201
    data = response.json()
    assert data["name"] == "Widget"
    assert data["price"] == 9.99
    assert "id" in data
    print(f"PASSED: happy path → id={data['id']}")

# ── Validation error (missing required field) ────────────────────────────────
def test_create_item_validation_error():
    reset_db()
    # 'price' is omitted — Pydantic will reject this
    response = client.post("/items", json={"name": "Widget"})
    assert response.status_code == 422
    body = response.json()
    # FastAPI wraps validation errors in {"detail": [...]}
    assert "detail" in body
    print("PASSED: missing field → 422 Unprocessable Entity")

# ── Duplicate item ───────────────────────────────────────────────────────────
def test_create_item_duplicate():
    reset_db()
    client.post("/items", json={"name": "Widget", "price": 9.99})
    response = client.post("/items", json={"name": "Widget", "price": 19.99})
    assert response.status_code == 409
    assert response.json()["detail"] == "Item already exists"
    print("PASSED: duplicate name → 409 Conflict")

test_create_item_happy()
test_create_item_validation_error()
test_create_item_duplicate()


### What just happened?
- **422 is free** — FastAPI raises it automatically whenever a request body fails Pydantic validation.
- **`reset_db()`** mimics what a pytest fixture's teardown does: starts each test with a clean slate.
- The duplicate test posts the same `name` twice and confirms the app enforces the business rule.
- In real projects you'd use a `pytest.fixture` with `yield` rather than a manual reset function.


## Step 4 · Dependency overrides — isolating the database

Production apps use `Depends(get_db)` to inject a database session.
Testing against the real database is slow and fragile — use `app.dependency_overrides` instead.

```
Normal flow:
  request → get_db() → real DB session → route handler

Test flow:
  request → get_test_db() → in-memory DB → route handler
```

`app.dependency_overrides` is a plain dict mapping the original callable to the replacement.
FastAPI checks this dict on every request — no monkey-patching needed.


In [ ]:
from fastapi import Depends

# ── Simulated production dependency ─────────────────────────────────────────
def get_db():
    """Production: yields a real DB session."""
    # In real code: yield SessionLocal(); session.close()
    yield {"connection": "real-db"}

# ── App that depends on get_db ───────────────────────────────────────────────
app2 = FastAPI()

@app2.get("/ping")
def ping(db=Depends(get_db)):
    # Normally you'd query db here
    return {"source": db["connection"]}

# ── Test override ────────────────────────────────────────────────────────────
test_db_store: Dict[int, dict] = {}

def get_test_db():
    """Test replacement: in-memory dict, no external connections."""
    yield {"connection": "test-db"}

# Register the override — swaps get_db for get_test_db in all routes
app2.dependency_overrides[get_db] = get_test_db

client2 = TestClient(app2)

def test_dependency_override():
    response = client2.get("/ping")
    assert response.status_code == 200
    # Route sees "test-db", not "real-db"
    assert response.json() == {"source": "test-db"}
    print("PASSED: dependency override replaced get_db with get_test_db")

test_dependency_override()

# Clean up — clear overrides so tests don't bleed into each other
app2.dependency_overrides.clear()
print("Overrides cleared")


### What just happened?
- **`app.dependency_overrides`** is the FastAPI escape hatch for test isolation — no mocking SQLAlchemy session internals.
- Overrides apply globally to that `app` instance; always clear them after the test with `.clear()`.
- The route function `ping` sees the test DB dictionary rather than a real connection.
- **Pattern tip:** In pytest, set overrides in `conftest.py` fixtures so they're applied and torn down consistently.


## Step 5 · pytest fixtures — setup and teardown

pytest **fixtures** are the standard way to prepare and clean up state around tests.
A fixture with `yield` divides into **setup** (before yield) and **teardown** (after yield).

```python
@pytest.fixture
def client():
    # --- setup ---
    app.dependency_overrides[get_db] = get_test_db
    with TestClient(app) as c:
        yield c          # test runs here
    # --- teardown ---
    app.dependency_overrides.clear()
```

Using `with TestClient(app) as c:` ensures lifespan events (`startup` / `shutdown`) fire during the test, matching production behaviour.


In [ ]:
import pytest

# ── Demonstrate fixture pattern inline (no conftest.py needed in Colab) ──────

class FixtureSimulator:
    """Manually replicate what a pytest fixture does so we can show it in-notebook."""

    def __enter__(self):
        # Setup: override dependency, reset store, create client
        fake_db.clear()
        globals()['next_id'] = 1
        app.dependency_overrides.clear()  # start clean
        self.client = TestClient(app)
        return self.client

    def __exit__(self, *args):
        # Teardown: clear overrides and store
        app.dependency_overrides.clear()
        fake_db.clear()
        print("Teardown complete — store cleared, overrides removed")

# ── Test using the fixture simulator ────────────────────────────────────────
def test_with_fixture_pattern():
    with FixtureSimulator() as c:
        # Create two items
        r1 = c.post("/items", json={"name": "Alpha", "price": 1.0})
        r2 = c.post("/items", json={"name": "Beta",  "price": 2.0})
        assert r1.status_code == 201
        assert r2.status_code == 201

        # Fetch Alpha by its assigned id
        item_id = r1.json()["id"]
        r3 = c.get(f"/items/{item_id}")
        assert r3.status_code == 200
        assert r3.json()["name"] == "Alpha"
        print("PASSED: fixture pattern — create + fetch lifecycle")

test_with_fixture_pattern()

# Verify teardown actually cleared state
print(f"fake_db after teardown: {fake_db}")


### What just happened?
- The `FixtureSimulator` context manager mirrors what `@pytest.fixture` with `yield` does: setup → test → teardown.
- **`fake_db.clear()`** in teardown prevents test pollution — each test starts with a blank store.
- In a real pytest project, this pattern lives in `conftest.py` and pytest auto-injects it via function arguments.
- **Key insight:** `with TestClient(app) as c:` triggers lifespan; plain `TestClient(app)` does not.


## Step 6 · Running pytest inline with pytest.main()

You can run `pytest.main()` directly inside a notebook cell to execute `.py` test files inline.
Here we write a test module to a temp file and run it — exactly what happens in CI.

This is the **review** step: we're checking all the patterns from this notebook still pass when wired into pytest.


In [ ]:
import pathlib

# Write a self-contained test module that pytest can discover
test_code = '''
import pytest
from fastapi import FastAPI, HTTPException, Depends
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Dict

# ── App under test ───────────────────────────────────────────────────────────
db: Dict[int, dict] = {}
counter = {"n": 1}

class ItemIn(BaseModel):
    name: str
    price: float

app = FastAPI()

@app.get("/")
def root():
    return {"ok": True}

@app.post("/items", status_code=201)
def create(item: ItemIn):
    for v in db.values():
        if v["name"] == item.name:
            raise HTTPException(409, "exists")
    rec = {"id": counter["n"], **item.dict()}
    db[counter["n"]] = rec
    counter["n"] += 1
    return rec

# ── Fixtures ─────────────────────────────────────────────────────────────────
@pytest.fixture(autouse=True)
def reset():
    db.clear()
    counter["n"] = 1
    yield
    db.clear()

@pytest.fixture
def client():
    with TestClient(app) as c:
        yield c

# ── Tests ────────────────────────────────────────────────────────────────────
def test_root(client):
    r = client.get("/")
    assert r.status_code == 200
    assert r.json() == {"ok": True}

def test_create_happy(client):
    r = client.post("/items", json={"name": "A", "price": 1.0})
    assert r.status_code == 201
    assert r.json()["name"] == "A"

def test_create_validation(client):
    r = client.post("/items", json={"name": "A"})  # missing price
    assert r.status_code == 422

def test_create_duplicate(client):
    client.post("/items", json={"name": "A", "price": 1.0})
    r = client.post("/items", json={"name": "A", "price": 2.0})
    assert r.status_code == 409
'''

test_file = pathlib.Path("/tmp/test_fastapi_day10.py")
test_file.write_text(test_code)
print(f"Test file written to {test_file}")

# Run pytest programmatically — exit code 0 = all passed
exit_code = pytest.main(["-v", str(test_file)])
print(f"\npytest exit code: {exit_code} ({'PASS' if exit_code == 0 else 'FAIL'})")


### What just happened?
- We wrote a complete pytest module to `/tmp/` and invoked `pytest.main()` — same as running `pytest -v` in the terminal.
- **`autouse=True`** on the `reset` fixture means pytest applies it to every test in the file automatically.
- The `client` fixture wraps `TestClient` in a `with` block, ensuring lifespan fires for every test.
- **Exit code 0** means all assertions passed; any non-zero value indicates a failure or error.


## Step 7 · Async tests with AsyncClient

For apps that use `async def` route handlers, `TestClient` still works (it runs an event loop internally).
But when you want **true async tests** — e.g., testing concurrent behaviour — use `httpx.AsyncClient` with `pytest-asyncio`.

```python
# conftest.py
import pytest

@pytest.fixture
async def async_client():
    async with AsyncClient(app=app, base_url="http://test") as ac:
        yield ac
```

Mark each async test with `@pytest.mark.asyncio` so pytest knows to schedule it on the event loop.


In [ ]:
import asyncio
import httpx
from fastapi import FastAPI

# ── Async app with an async route handler ────────────────────────────────────
async_app = FastAPI()

@async_app.get("/async-ping")
async def async_ping():
    # Simulate an async DB call
    await asyncio.sleep(0)  # yields control, like awaiting a real DB query
    return {"pong": True, "method": "async"}

@async_app.post("/async-echo")
async def async_echo(body: dict):
    return {"echoed": body}

# ── Async test using AsyncClient (equivalent to pytest.mark.asyncio) ─────────
async def run_async_tests():
    # httpx.AsyncClient with ASGITransport replaces the deprecated app= kwarg
    transport = httpx.ASGITransport(app=async_app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as ac:
        # Test 1: async GET
        r = await ac.get("/async-ping")
        assert r.status_code == 200
        assert r.json()["pong"] is True
        print("PASSED: async GET /async-ping")

        # Test 2: async POST with JSON body
        r2 = await ac.post("/async-echo", json={"key": "value"})
        assert r2.status_code == 200
        assert r2.json() == {"echoed": {"key": "value"}}
        print("PASSED: async POST /async-echo")

# asyncio.run() drives the event loop in Colab
asyncio.run(run_async_tests())


### What just happened?
- **`httpx.ASGITransport`** is the modern way to connect AsyncClient to an ASGI app (replaces deprecated `app=` kwarg).
- We `await` each `.get()` / `.post()` call — the requests flow through the async event loop, not a background thread.
- In a real pytest project, decorate each async test with `@pytest.mark.asyncio` and pytest-asyncio handles the `asyncio.run()` for you.
- **When to prefer AsyncClient:** when testing WebSocket handshakes, streaming responses, or concurrent request patterns.


## Step 8 · Reviewing all patterns: dependency override + pytest.main() together

Now we review all patterns together in one complete test file:
- Fixture with dependency override
- autouse reset fixture
- happy path, validation error, not-found, and duplicate tests
- Run via `pytest.main()`

This mirrors a production `tests/test_items.py` file structure.


In [ ]:
full_test_code = '''
import pytest
from fastapi import FastAPI, HTTPException, Depends
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Dict, Generator

# ── Production dependency ─────────────────────────────────────────────────────
class FakeProductionDB:
    data: Dict[int, dict] = {}
    next_id: int = 1

prod_db = FakeProductionDB()

def get_db() -> Generator:
    yield prod_db  # would yield a SQLAlchemy Session in production

# ── App under test ────────────────────────────────────────────────────────────
class ItemIn(BaseModel):
    name: str
    price: float

app = FastAPI()

@app.get("/")
def root():
    return {"status": "ok"}

@app.post("/items", status_code=201)
def create_item(item: ItemIn, db=Depends(get_db)):
    for v in db.data.values():
        if v["name"] == item.name:
            raise HTTPException(409, "duplicate")
    rec = {"id": db.next_id, "name": item.name, "price": item.price}
    db.data[db.next_id] = rec
    db.next_id += 1
    return rec

@app.get("/items/{item_id}")
def get_item(item_id: int, db=Depends(get_db)):
    if item_id not in db.data:
        raise HTTPException(404, "not found")
    return db.data[item_id]

# ── Test DB + Fixtures ────────────────────────────────────────────────────────
class TestDB:
    data: Dict[int, dict] = {}
    next_id: int = 1

test_db_instance = TestDB()

def get_test_db() -> Generator:
    yield test_db_instance

@pytest.fixture(autouse=True)
def reset_test_db():
    test_db_instance.data.clear()
    test_db_instance.next_id = 1
    yield
    test_db_instance.data.clear()

@pytest.fixture
def client():
    app.dependency_overrides[get_db] = get_test_db
    with TestClient(app) as c:
        yield c
    app.dependency_overrides.clear()

# ── Tests ─────────────────────────────────────────────────────────────────────
def test_root(client):
    assert client.get("/").json() == {"status": "ok"}

def test_create_happy(client):
    r = client.post("/items", json={"name": "X", "price": 5.0})
    assert r.status_code == 201
    assert r.json()["id"] == 1

def test_create_validation_error(client):
    r = client.post("/items", json={"name": "X"})  # price missing
    assert r.status_code == 422

def test_create_duplicate(client):
    client.post("/items", json={"name": "X", "price": 1.0})
    r = client.post("/items", json={"name": "X", "price": 2.0})
    assert r.status_code == 409

def test_get_not_found(client):
    r = client.get("/items/999")
    assert r.status_code == 404

def test_get_existing(client):
    created = client.post("/items", json={"name": "Y", "price": 3.0}).json()
    r = client.get(f"/items/{created[\"id\"]}")  
    assert r.status_code == 200
    assert r.json()["name"] == "Y"
'''

full_test_file = pathlib.Path("/tmp/test_day10_full.py")
full_test_file.write_text(full_test_code)

exit_code = pytest.main(["-v", "--tb=short", str(full_test_file)])
print(f"\nAll tests exit code: {exit_code}")


### What just happened?
- All patterns from the day are combined: dependency override isolates the DB, `autouse=True` reset keeps tests independent.
- **`app.dependency_overrides.clear()`** in fixture teardown ensures no override leaks to other test modules.
- Six tests covering root, happy path, validation error, duplicate, 404, and successful retrieval.
- **Separation of concerns:** test DB is a separate `TestDB` class, not the global `fake_db` — cleaner than patching globals.


In [ ]:
# Challenge: Extend the test suite
#
# Your task: add the following 3 tests to the test file above:
#
# 1. test_create_multiple_items — create 3 items, assert each gets a unique id
# 2. test_price_validation — POST with price="not-a-number", assert 422
# 3. test_async_root — use httpx.AsyncClient + asyncio.run() to GET / asynchronously
#
# Scaffold:

import httpx, asyncio
from fastapi.testclient import TestClient

# Reuse the app defined earlier in this notebook
challenge_client = TestClient(app)

def test_create_multiple_items():
    reset_db()
    ids = []
    for i in range(3):
        r = challenge_client.post("/items", json={"name": f"item-{i}", "price": float(i)})
        # YOUR ASSERTION HERE: check status and collect id
        pass
    # YOUR ASSERTION HERE: verify all ids are unique
    pass

def test_price_validation():
    r = challenge_client.post("/items", json={"name": "bad", "price": "not-a-number"})
    # YOUR ASSERTION HERE
    pass

async def _async_root():
    transport = httpx.ASGITransport(app=app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as ac:
        r = await ac.get("/")
        # YOUR ASSERTION HERE
        return r.json()

def test_async_root():
    result = asyncio.run(_async_root())
    # YOUR ASSERTION HERE
    pass

# Run your tests:
# test_create_multiple_items()
# test_price_validation()
# test_async_root()
print("Implement the assertions above, then uncomment the calls!")


---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| `TestClient` | In-process ASGI client — fast, no network socket, lifespan fires with `with` |
| `app.dependency_overrides` | Dict mapping real → test callable; always `.clear()` in teardown |
| pytest fixture | `@pytest.fixture` with `yield` = setup / teardown; `autouse=True` = auto-applied |
| 422 Unprocessable Entity | FastAPI/Pydantic raises automatically for schema violations — no custom handler needed |
| `AsyncClient` | For async test routes; use `httpx.ASGITransport(app=app)` + `await` |
| `pytest.main()` | Run pytest programmatically from a script or notebook |

> **Tip:** Use `app.dependency_overrides` to replace `get_db` with `get_test_db` — isolated tests without mocking SQLAlchemy.

---
## What's next
**Day 11** → File Uploads and Form Data — accept `UploadFile`, validate MIME types, stream large files, and mix form fields with file uploads in one endpoint.

Mark Day 10 complete in your [tracker](../index.html).
